In [1]:
import json
from bs4 import BeautifulSoup
import re

def clean_text(text):
    """Loại bỏ chú thích dạng [1], [2], các ký tự đặc biệt và khoảng trắng thừa."""
    text = re.sub(r'\[\d+\]', '', text)
    return ' '.join(text.split())

def transform_to_clean_json(input_file, output_file):
    print(f"--- Đang xử lý file: {input_file} ---")
    
    with open(input_file, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    clean_system = {
        "nhan_vat": raw_data.get("nhan_vat", "Không rõ"),
        "url_nguon": raw_data.get("url_chinh", ""),
        "tieu_su_he_thong": []
    }

    # Duyệt qua từng mục HTML chi tiết trong file gốc
    for detail in raw_data.get("chi_tiet", []):
        muc_title = detail.get("muc", "Không có tiêu đề")
        html_content = detail.get("html", "")
        
        soup = BeautifulSoup(html_content, 'html.parser')
        
        # Loại bỏ các thẻ không chứa nội dung văn bản hữu ích
        for noise in soup(['script', 'style', 'table', 'nav', 'header', 'footer', 'sup']):
            noise.decompose()

        # Trích xuất nội dung từ các thẻ <p> (đoạn văn)
        paragraphs = []
        for p in soup.find_all('p'):
            p_text = clean_text(p.get_text())
            if p_text and len(p_text) > 10: # Chỉ lấy các đoạn có nghĩa
                paragraphs.append(p_text)

        # Chỉ thêm vào JSON nếu mục đó thực sự có nội dung văn bản
        if paragraphs:
            clean_system["tieu_su_he_thong"].append({
                "muc": muc_title,
                "url_chi_tiet": detail.get("url_muc"),
                "noi_dung": paragraphs
            })

    # Xuất ra file JSON sạch
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(clean_system, f, ensure_ascii=False, indent=4)
    
    print(f"--- Hoàn tất! Đã xuất file sạch tại: {output_file} ---")

# Thực hiện chuyển đổi
transform_to_clean_json('Đinh Tiên Hoàng_system.json', 'Dinh_Tien_Hoang_Clean.json')

--- Đang xử lý file: Đinh Tiên Hoàng_system.json ---
--- Hoàn tất! Đã xuất file sạch tại: Dinh_Tien_Hoang_Clean.json ---
